# 案例 04 · 安全组误删，VPC 内调用超时

> 沙箱实验：真实进程 + 阿里云 OpenAPI 模拟（响应带 `Mock: true`）。
> 云产品字段/命令与生产一致，资源规格与账单为模拟值。

## 场景

一次安全组收敛变更误删了 note-api 端口的入方向规则，VPC 内上游调用出现 TCP 连接超时（SYN 无响应、无 RST），实例本身运行正常。

## 练习目标

- 用 `tcp_probe` 观察“超时”而非“拒绝”
- 对比安全组规则与基线
- 以最小权限补回规则并复测
- 用 `sg_audit.py` 完成自动化处置

## 前置

- 仓库根目录执行 `python -m venv .venv && .venv/bin/pip install -r requirements.txt`
- 使用该 venv 的 Python kernel 打开本 Notebook；或运行 `tools/run_all_labs.py` 批量执行


In [ ]:
# 环境准备：定位仓库根目录并加载沙箱
import json, os, signal, ssl, socket, sys, time, gzip, shutil, urllib.parse, urllib.request
from pathlib import Path

REPO = Path.cwd().resolve()
while not (REPO / "labs" / "common" / "xhs_lab.py").exists():
    if REPO.parent == REPO:
        raise RuntimeError("请在 xhs-backend-cloud-ops-lab 仓库内运行本 Notebook")
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

from labs.common import Lab

lab = Lab("case-04-sg-timeout")
lab.reset(start_api=True)

lab.summary()


## 步骤 1 · 异常重现

下面注入一个可复现的真实故障，观察它对业务的影响。

In [ ]:
lab.inject_sg_misconfig()
print("已注入：删除 api 端口的安全组入方向规则")


## 步骤 2 · 取证

按“入口 → 主机 → 进程 → 依赖 → 日志”的分层顺序收集证据，不要一上来就重启。

**安全组规则**

In [ ]:
print(lab.aliyun("ecs DescribeSecurityGroups --SecurityGroupId sg-bp1xhsapi"))

**连通性探测**

In [ ]:
print(lab.tcp_probe("10.10.1.11", lab.ports["api"]))

**实例状态**

In [ ]:
print(lab.aliyun("ecs DescribeInstances"))

## 步骤 3 · 判断

把证据收敛成一句结论，决定止血动作。

In [ ]:
sg = lab.state["security_groups"][0]
ports = [r["PortRange"] for r in sg["Rules"]]
print("入方向端口:", ports)
print("业务端口:", lab.ports["api"])
print("探测结果:", lab.tcp_probe("10.10.1.11", lab.ports["api"]))
print("判断：timeout（无 RST）通常指向安全组/网络 ACL 丢包，而非进程未监听")


## 步骤 4 · 手工处置（先止血）

执行最小必要的变更，优先恢复业务，再考虑根治。

In [ ]:
sg = lab.state["security_groups"][0]
port = lab.ports["api"]
# 最小权限：仅放通 VPC 网段到业务端口
lab.aliyun(f"ecs AuthorizeSecurityGroup --SecurityGroupId {sg['SecurityGroupId']} "
           f"--IpProtocol tcp --PortRange {port}/{port} --SourceCidrIp 10.10.0.0/16 "
           f"--Description xhs-api-vpc")
lab.state["faults"].pop("network", None)
lab._save()
print(lab.tcp_probe("10.10.1.11", port))


## 步骤 5 · 验证

用可执行的断言确认恢复，而不是“看起来好了”。

In [ ]:
lab.check('端口已放通', bool(lab.tcp_probe('10.10.1.11', lab.ports['api'])['result'] == 'open'))
lab.check('规则存在', bool(any(r['PortRange'].startswith(str(lab.ports['api'])) for r in lab.state['security_groups'][0]['Rules'])))

## 步骤 6 · 自动化沉淀

脚本把安全组规则与基线比对，找出缺口后按最小权限提交 `AuthorizeSecurityGroup`，并用策略模拟器复测连通性。

下面重置沙箱、重新注入故障，然后直接调用生产可用的自动化脚本（`--apply`；缺省为 dry-run）。

In [ ]:
lab.reset(start_api=True)
lab.inject_sg_misconfig()
out = lab.auto('sg_audit.py', '--apply')
print(out)

In [ ]:
lab.check('自动化处置返回 REMEDIATED', 'RESULT: REMEDIATED' in out)
print('\n自动化脚本已可作为 cron / 事件总线触发器复用')

## 复盘与思考题

1. 连接超时与连接拒绝分别说明什么？
2. 安全组变更如何做审批与回滚？
3. 为什么安全组规则要按“网段 + 端口”最小化，而不是全放通？

## 收尾

沙箱资源会在本 Cell 中停止；重新运行请从第 1 个代码 Cell 开始。


In [ ]:
lab.stop_all()
print('沙箱已停止')